# Pakistani Political Members Neo4j Graph - GraphRAG with LangGraph

This notebook demonstrates:
1. Creating a Neo4j knowledge graph of Pakistani political members
2. Populating it with political data
3. Querying it using LangGraph with natural language
4. Visualizing the relationships and network

## 1. Import Required Libraries

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'neo4j>=5.14.0',
    'langgraph>=0.0.60',
    'langchain>=0.1.0',
    'langchain-core>=0.1.0',
    'openai>=1.0.0',
    'python-dotenv>=1.0.0',
    'pandas>=2.0.0',
    'networkx>=3.0',
    'matplotlib>=3.5.0',
]

for package in packages:
    try:
        __import__(package.split('>=')[0])
        print(f'✓ {package.split(">")[0]} already installed')
    except ImportError:
        print(f'Installing {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [ ]:
# Import required libraries
import os
import json
from typing import Any, Optional
from neo4j import GraphDatabase, Session
from dotenv import load_dotenv
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from datetime import datetime

# Load environment variables
load_dotenv()

print('✓ All libraries imported successfully')

## 2. Connect to Neo4j Database

In [ ]:
# Configuration
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

print(f'Connecting to Neo4j at {NEO4J_URI}...')

try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print('✓ Successfully connected to Neo4j')
except Exception as e:
    print(f'✗ Error connecting to Neo4j: {e}')
    print('Make sure Neo4j is running. You can start it with:')
    print('docker run -d -p 7687:7687 -p 7474:7474 -e NEO4J_AUTH=neo4j/password neo4j:5.14-community')

## 3. Create Graph Schema for Political Members

In [ ]:
# Function to create indexes and clear database
def setup_graph_schema(driver):
    """Setup the graph schema with indexes and constraints."""
    with driver.session() as session:
        # Clear existing data
        print('Clearing existing data...')
        session.run('MATCH (n) DETACH DELETE n')
        
        # Create indexes
        print('Creating indexes...')
        session.run('CREATE INDEX IF NOT EXISTS FOR (p:Person) ON (p.id)')
        session.run('CREATE INDEX IF NOT EXISTS FOR (p:Person) ON (p.name)')
        session.run('CREATE INDEX IF NOT EXISTS FOR (p:Party) ON (p.name)')
        session.run('CREATE INDEX IF NOT EXISTS FOR (p:Position) ON (p.title)')
        
        print('✓ Graph schema created with indexes')
        
        # Print schema information
        print('\nNode Labels and Relationship Types will be created as data is added:')
        print('  Node Labels: Person, Party, Position')
        print('  Relationships: MEMBER_OF, HELD_POSITION, COLLABORATED_WITH')

# Setup the schema
setup_graph_schema(driver)

## 4. Add Pakistani Political Members to Neo4j

In [ ]:
# Define Pakistani political members
political_members = [
    {
        'id': 'benazir_bhutto',
        'name': 'Benazir Bhutto',
        'birth_year': 1953,
        'bio': 'First female Prime Minister of Pakistan (1988-1990)',
        'status': 'Deceased'
    },
    {
        'id': 'nawaz_sharif',
        'name': 'Muhammad Nawaz Sharif',
        'birth_year': 1949,
        'bio': 'Former Prime Minister and business tycoon, leader of PMLN',
        'status': 'Active'
    },
    {
        'id': 'imran_khan',
        'name': 'Imran Khan',
        'birth_year': 1952,
        'bio': 'Former cricketer, founder and chairman of Pakistan Tehreek-e-Insaaf',
        'status': 'Active'
    },
    {
        'id': 'shehbaz_sharif',
        'name': 'Shehbaz Sharif',
        'birth_year': 1951,
        'bio': 'Current Prime Minister and PMLN member, brother of Nawaz Sharif',
        'status': 'Active'
    },
    {
        'id': 'asif_ali_zardari',
        'name': 'Asif Ali Zardari',
        'birth_year': 1955,
        'bio': 'Former President of Pakistan, PPP member, widower of Benazir Bhutto',
        'status': 'Active'
    },
    {
        'id': 'bilawal_bhutto',
        'name': 'Bilawal Bhutto Zardari',
        'birth_year': 1988,
        'bio': 'Current Foreign Minister of Pakistan, Chairman of PPP, son of Benazir Bhutto',
        'status': 'Active'
    },
    {
        'id': 'maryam_nawaz',
        'name': 'Maryam Nawaz Sharif',
        'birth_year': 1973,
        'bio': 'Vice President of PMLN, daughter of Nawaz Sharif',
        'status': 'Active'
    },
    {
        'id': 'raja_ashraf',
        'name': 'Raja Pervaiz Ashraf',
        'birth_year': 1954,
        'bio': 'Former Prime Minister of Pakistan (2008-2012), PPP member',
        'status': 'Active'
    },
    {
        'id': 'shahid_khaqan',
        'name': 'Shahid Khaqan Abbasi',
        'birth_year': 1958,
        'bio': 'Former Prime Minister of Pakistan (2017-2018), PMLN member',
        'status': 'Active'
    },
    {
        'id': 'hafiz_sheikh',
        'name': 'Muhammad Hafeez Sheikh',
        'birth_year': 1955,
        'bio': 'Finance Minister of Pakistan, economic expert',
        'status': 'Active'
    },
    {
        'id': 'fawad_chaudhry',
        'name': 'Fawad Chaudhry',
        'birth_year': 1970,
        'bio': 'Minister of Information, PTI member, digital expert',
        'status': 'Active'
    },
]

# Add political members to Neo4j
def add_political_members(driver, members):
    with driver.session() as session:
        for member in members:
            session.run(
                """
                MERGE (p:Person {id: $id})
                SET p.name = $name, p.birth_year = $birth_year, p.bio = $bio, p.status = $status
                """,
                id=member['id'],
                name=member['name'],
                birth_year=member['birth_year'],
                bio=member['bio'],
                status=member['status']
            )
    print(f'✓ Added {len(members)} political members to Neo4j')
    return len(members)

members_added = add_political_members(driver, political_members)
print(f'\nTotal members added: {members_added}')

In [ ]:
# Define political parties
parties = [
    {'id': 'ppp', 'name': 'Pakistan Peoples Party', 'founded': 1967, 'ideology': 'Center-left'},
    {'id': 'pmln', 'name': 'Pakistan Muslim League (Nawaz)', 'founded': 1997, 'ideology': 'Center-right'},
    {'id': 'ptm', 'name': 'Pakistan Tehreek-e-Insaaf', 'founded': 1996, 'ideology': 'Center-right'},
    {'id': 'jui', 'name': 'Jamiat Ulema-e-Islam', 'founded': 1945, 'ideology': 'Religious'},
    {'id': 'mqm', 'name': 'Muttahida Qaumi Movement', 'founded': 1984, 'ideology': 'Ethnic'},
]

# Add parties to Neo4j
def add_parties(driver, parties):
    with driver.session() as session:
        for party in parties:
            session.run(
                """
                MERGE (p:Party {id: $id})
                SET p.name = $name, p.founded = $founded, p.ideology = $ideology
                """,
                id=party['id'],
                name=party['name'],
                founded=party['founded'],
                ideology=party['ideology']
            )
    print(f'✓ Added {len(parties)} political parties to Neo4j')

add_parties(driver, parties)

# Define political positions
positions = [
    {'id': 'pm', 'title': 'Prime Minister', 'level': 'National'},
    {'id': 'president', 'title': 'President', 'level': 'National'},
    {'id': 'fm', 'title': 'Foreign Minister', 'level': 'National'},
    {'id': 'finance_min', 'title': 'Finance Minister', 'level': 'National'},
    {'id': 'cm', 'title': 'Chief Minister', 'level': 'Provincial'},
]

# Add positions to Neo4j
def add_positions(driver, positions):
    with driver.session() as session:
        for position in positions:
            session.run(
                """
                MERGE (pos:Position {id: $id})
                SET pos.title = $title, pos.level = $level
                """,
                id=position['id'],
                title=position['title'],
                level=position['level']
            )
    print(f'✓ Added {len(positions)} political positions to Neo4j')

add_positions(driver, positions)

## 5. Define Relationships Between Political Members

In [ ]:
# Define party memberships
memberships = [
    ('benazir_bhutto', 'ppp', 1982, 2007),
    ('nawaz_sharif', 'pmln', 1997, None),
    ('imran_khan', 'ptm', 1996, None),
    ('shehbaz_sharif', 'pmln', 1997, None),
    ('asif_ali_zardari', 'ppp', 1990, None),
    ('bilawal_bhutto', 'ppp', 2010, None),
    ('maryam_nawaz', 'pmln', 2002, None),
    ('raja_ashraf', 'ppp', 1980, None),
    ('shahid_khaqan', 'pmln', 2000, None),
    ('hafiz_sheikh', 'pmln', 2005, None),
    ('fawad_chaudhry', 'ptm', 2014, None),
]

# Create MEMBER_OF relationships
def create_memberships(driver, memberships):
    with driver.session() as session:
        for person_id, party_id, start_year, end_year in memberships:
            session.run(
                """
                MATCH (p:Person {id: $person_id}), (party:Party {id: $party_id})
                MERGE (p)-[r:MEMBER_OF]->(party)
                SET r.start_year = $start_year, r.end_year = $end_year
                """,
                person_id=person_id,
                party_id=party_id,
                start_year=start_year,
                end_year=end_year
            )
    print(f'✓ Created {len(memberships)} party membership relationships')

create_memberships(driver, memberships)

In [ ]:
# Define position holdings
position_holdings = [
    ('benazir_bhutto', 'pm', 1988, 1990),
    ('nawaz_sharif', 'pm', 1990, 1993),
    ('nawaz_sharif', 'pm', 1997, 1999),
    ('imran_khan', 'pm', 2018, 2022),
    ('shehbaz_sharif', 'pm', 2022, None),
    ('bilawal_bhutto', 'fm', 2023, None),
    ('raja_ashraf', 'pm', 2008, 2012),
    ('shahid_khaqan', 'pm', 2017, 2018),
    ('hafiz_sheikh', 'finance_min', 2022, None),
    ('asif_ali_zardari', 'president', 2008, 2013),
]

# Create HELD_POSITION relationships
def create_position_holdings(driver, holdings):
    with driver.session() as session:
        for person_id, pos_id, start_year, end_year in holdings:
            session.run(
                """
                MATCH (p:Person {id: $person_id}), (pos:Position {id: $pos_id})
                MERGE (p)-[r:HELD_POSITION]->(pos)
                SET r.start_year = $start_year, r.end_year = $end_year
                """,
                person_id=person_id,
                pos_id=pos_id,
                start_year=start_year,
                end_year=end_year
            )
    print(f'✓ Created {len(holdings)} position holding relationships')

create_position_holdings(driver, position_holdings)

In [ ]:
# Define collaborations/relationships
collaborations = [
    ('nawaz_sharif', 'shehbaz_sharif', 'Brothers and political allies'),
    ('maryam_nawaz', 'nawaz_sharif', 'Father and daughter political team'),
    ('asif_ali_zardari', 'benazir_bhutto', 'Husband and wife (deceased spouse)'),
    ('bilawal_bhutto', 'asif_ali_zardari', 'Son and father in same party'),
    ('imran_khan', 'fawad_chaudhry', 'Political allies in PTI'),
    ('bilawal_bhutto', 'benazir_bhutto', 'Son and mother'),
    ('raja_ashraf', 'benazir_bhutto', 'Political colleagues in PPP'),
]

# Create COLLABORATED_WITH relationships
def create_collaborations(driver, collaborations):
    with driver.session() as session:
        for person_id_1, person_id_2, description in collaborations:
            session.run(
                """
                MATCH (p1:Person {id: $person_id_1}), (p2:Person {id: $person_id_2})
                MERGE (p1)-[r:COLLABORATED_WITH]->(p2)
                SET r.description = $description
                """,
                person_id_1=person_id_1,
                person_id_2=person_id_2,
                description=description
            )
    print(f'✓ Created {len(collaborations)} collaboration relationships')

create_collaborations(driver, collaborations)

In [ ]:
# Check graph statistics
def get_graph_stats(driver):
    with driver.session() as session:
        people = session.run('MATCH (p:Person) RETURN count(p) as count').single()['count']
        parties = session.run('MATCH (p:Party) RETURN count(p) as count').single()['count']
        positions = session.run('MATCH (p:Position) RETURN count(p) as count').single()['count']
        relationships = session.run('MATCH ()-[r]->() RETURN count(r) as count').single()['count']
        
        return {
            'People': people,
            'Parties': parties,
            'Positions': positions,
            'Relationships': relationships
        }

stats = get_graph_stats(driver)
print('\n📊 Graph Statistics:')
for key, value in stats.items():
    print(f'  {key}: {value}')

## 6. Build LangGraph Query Agent

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# Define state for the graph
class QueryState(TypedDict):
    """State for the political graph query agent."""
    messages: list[BaseMessage]
    query: str
    neo4j_results: dict[str, Any]
    response: str

print('✓ LangGraph state and imports configured')

In [ ]:
# Create a Neo4j query handler
class PoliticalGraphQuerier:
    """Handles queries to the political graph."""
    
    def __init__(self, driver):
        self.driver = driver
    
    def get_prime_ministers(self):
        """Get all Prime Ministers."""
        with self.driver.session() as session:
            result = session.run(
                """
                MATCH (p:Person)-[r:HELD_POSITION]->(pos:Position)
                WHERE pos.title = 'Prime Minister'
                RETURN p.name as name, r.start_year as start_year, r.end_year as end_year
                ORDER BY r.start_year DESC
                """
            )
            return [dict(record) for record in result]
    
    def get_party_members(self, party_name):
        """Get members of a specific party."""
        with self.driver.session() as session:
            result = session.run(
                """
                MATCH (p:Person)-[:MEMBER_OF]->(party:Party)
                WHERE party.name CONTAINS $party_name
                RETURN p.name as name, p.birth_year as birth_year
                ORDER BY p.name
                """,
                party_name=party_name
            )
            return [dict(record) for record in result]
    
    def get_person_info(self, name):
        """Get detailed info about a person."""
        with self.driver.session() as session:
            person_result = session.run(
                """MATCH (p:Person) WHERE p.name CONTAINS $name RETURN p.name as name, p.bio as bio, p.birth_year as birth_year LIMIT 1""",
                name=name
            ).single()
            
            if not person_result:
                return None
            
            person_data = dict(person_result)
            
            # Get positions
            positions = session.run(
                """MATCH (p:Person)-[r:HELD_POSITION]->(pos:Position) WHERE p.name CONTAINS $name 
                   RETURN pos.title as title, r.start_year as start_year, r.end_year as end_year""",
                name=name
            )
            person_data['positions'] = [dict(p) for p in positions]
            
            return person_data
    
    def get_collaborators(self, name):
        """Get collaborators of a person."""
        with self.driver.session() as session:
            result = session.run(
                """MATCH (p:Person)-[r:COLLABORATED_WITH]->(other:Person) WHERE p.name CONTAINS $name
                   RETURN other.name as name, r.description as description""",
                name=name
            )
            return [dict(record) for record in result]
    
    def get_network_overview(self):
        """Get an overview of the political network."""
        with self.driver.session() as session:
            result = session.run(
                """
                MATCH (p:Person)-[:MEMBER_OF]->(party:Party)
                RETURN party.name as party, count(p) as member_count
                ORDER BY member_count DESC
                """
            )
            return [dict(record) for record in result]

querier = PoliticalGraphQuerier(driver)
print('✓ PoliticalGraphQuerier initialized')

In [ ]:
# Create LangGraph agent nodes
def process_query_node(state: QueryState) -> Command:
    """Process the user query and fetch relevant data from Neo4j."""
    query = state.get('query', '').lower()
    results = {}
    
    # Analyze query and fetch appropriate data
    if 'prime minister' in query:
        results['prime_ministers'] = querier.get_prime_ministers()
    
    if 'member' in query and any(p in query for p in ['ppp', 'pmln', 'ptm', 'party']):
        for party in ['PPP', 'PMLN', 'PTI']:
            if party.lower() in query:
                results[f'party_members'] = querier.get_party_members(party)
                break
    
    if 'who is' in query or 'about' in query:
        words = query.split()
        for i, word in enumerate(words):
            if word in ['is', 'about']:
                if i + 1 < len(words):
                    potential_name = ' '.join(words[i+1:i+3])
                    person_info = querier.get_person_info(potential_name)
                    if person_info:
                        results['person_info'] = person_info
                        results['collaborators'] = querier.get_collaborators(potential_name)
                    break
    
    if 'network' in query or 'overview' in query or 'statistics' in query:
        results['network_overview'] = querier.get_network_overview()
    
    # If no specific queries matched, get overview
    if not results:
        results['network_overview'] = querier.get_network_overview()
    
    state['neo4j_results'] = results
    return Command(goto='format_response')

def format_response_node(state: QueryState) -> Command:
    """Format the Neo4j results into a human-readable response."""
    results = state.get('neo4j_results', {})
    query = state.get('query', '')
    
    response = f"Query: {query}\n\nResults:\n"
    
    if 'prime_ministers' in results:
        response += "\n📋 Prime Ministers of Pakistan:\n"
        for pm in results['prime_ministers']:
            response += f"  • {pm['name']} ({pm['start_year']}-{pm['end_year']})\n"
    
    if 'party_members' in results:
        response += "\n👥 Party Members:\n"
        for member in results['party_members']:
            response += f"  • {member['name']} (b. {member['birth_year']})\n"
    
    if 'person_info' in results:
        info = results['person_info']
        response += f"\n🧑 {info['name']}\n"
        response += f"  Born: {info['birth_year']}\n"
        response += f"  Bio: {info['bio']}\n"
        if info.get('positions'):
            response += "  Positions held:\n"
            for pos in info['positions']:
                response += f"    - {pos['title']} ({pos['start_year']}-{pos['end_year']})\n"
    
    if 'collaborators' in results and results['collaborators']:
        response += "\n🤝 Collaborators:\n"
        for collab in results['collaborators']:
            response += f"  • {collab['name']}: {collab['description']}\n"
    
    if 'network_overview' in results:
        response += "\n🌐 Political Network Overview:\n"
        for party in results['network_overview']:
            response += f"  • {party['party']}: {party['member_count']} members\n"
    
    state['response'] = response
    return Command(goto=END)

print('✓ LangGraph nodes created')

In [ ]:
# Build the LangGraph
graph_builder = StateGraph(QueryState)

# Add nodes
graph_builder.add_node('process_query', process_query_node)
graph_builder.add_node('format_response', format_response_node)

# Set up edges
graph_builder.add_edge(START, 'process_query')

# Compile the graph
query_graph = graph_builder.compile()

print('✓ LangGraph compiled successfully')

## 7. Query the Graph Using LangGraph

In [ ]:
# Function to execute queries
def query_political_graph(question):
    """Execute a query against the political graph."""
    initial_state = {
        'messages': [],
        'query': question,
        'neo4j_results': {},
        'response': ''
    }
    
    final_state = query_graph.invoke(initial_state)
    return final_state.get('response', 'No response generated')

# Test queries
test_queries = [
    "Who are the Prime Ministers of Pakistan?",
    "Tell me about members of PPP",
    "What is the political network overview?",
    "Who is Benazir Bhutto?",
]

print("🤖 Testing LangGraph Query Agent\n")
print("="*70)

for query in test_queries[:2]:
    print(f"\n❓ Query: {query}")
    print("-"*70)
    response = query_political_graph(query)
    print(response)
    print()

In [ ]:
# More queries
print("\n" + "="*70)
for query in test_queries[2:]:
    print(f"\n❓ Query: {query}")
    print("-"*70)
    response = query_political_graph(query)
    print(response)
    print()

## 8. Visualize Query Results

In [ ]:
# Get all relationships for visualization
def get_graph_data():
    """Get graph data for visualization."""
    with driver.session() as session:
        # Get all people and parties
        people_result = session.run('MATCH (p:Person) RETURN p.name as name')
        people = [record['name'] for record in people_result]
        
        # Get collaboration relationships
        edges_result = session.run(
            'MATCH (p1:Person)-[r:COLLABORATED_WITH]->(p2:Person) RETURN p1.name as source, p2.name as target'
        )
        edges = [(record['source'], record['target']) for record in edges_result]
        
        return people, edges

people, edges = get_graph_data()
print(f'Graph has {len(people)} people and {len(edges)} collaboration edges')

In [ ]:
# Create and visualize the network graph
G = nx.DiGraph()
G.add_nodes_from(people)
G.add_edges_from(edges)

# Create visualization
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=2, iterations=50)

# Draw the network
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1500)
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15, alpha=0.6)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')

plt.title('Pakistani Political Members Collaboration Network', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'Network graph visualized with {len(G.nodes)} nodes and {len(G.edges)} edges')

In [ ]:
# Create a table of all members
with driver.session() as session:
    result = session.run(
        """
        MATCH (p:Person)
        OPTIONAL MATCH (p)-[r:MEMBER_OF]->(party:Party)
        OPTIONAL MATCH (p)-[pos_rel:HELD_POSITION]->(position:Position)
        RETURN 
            p.name as Name,
            p.birth_year as 'Birth Year',
            party.name as Party,
            position.title as Position
        ORDER BY p.name
        """
    )
    
    data = []
    for record in result:
        data.append({
            'Name': record['Name'],
            'Birth Year': record['Birth Year'],
            'Party': record['Party'] or '-',
            'Position': record['Position'] or '-'
        })

df = pd.DataFrame(data)
print('\n📊 Pakistani Political Members Summary\n')
print(df.to_string(index=False))

In [ ]:
# Summary statistics
print('\n' + '='*70)
print('📈 SUMMARY STATISTICS')
print('='*70)

with driver.session() as session:
    # Prime ministers count
    pms = session.run(
        "MATCH (p:Person)-[r:HELD_POSITION]->(pos:Position {title: 'Prime Minister'}) RETURN count(DISTINCT p.name) as count"
    ).single()['count']
    
    # Members by party
    members_by_party = session.run(
        """
        MATCH (p:Person)-[:MEMBER_OF]->(party:Party)
        RETURN party.name as party, count(p) as count
        ORDER BY count DESC
        """
    )
    
    print(f'\n👥 Total Political Members: {len(people)}')
    print(f'🏛️  Prime Ministers: {pms}')
    print(f'🎗️  Political Parties: {len(set(edges))}')
    print(f'\n📊 Members by Party:')
    for record in members_by_party:
        print(f'   • {record["party"]}: {record["count"]}')

print('\n' + '='*70)
print('✅ Analysis Complete!')
print('='*70)

In [ ]:
# Close the driver
driver.close()
print('✓ Neo4j connection closed')